In [ ]:
import numpy as np
from numpy.linalg import norm, solve, LinAlgError
import matplotlib.pyplot as plt
from case_studies import * 

In [ ]:
def solve_sphere_constrained(Q, b, Delta, tol=1e-8, max_iter=100):
    """
    Solve the quadratic minimization problem
      min (1/2 x^T Q x + b^T x)
      s.t. ||x||^2 <= Delta^2,
    using a bisection search on the Lagrange multiplier lambda.
    
    Returns:
      x_star: the computed solution,
      lambda_star: the final Lagrange multiplier,
      iter_count: number of bisection iterations,
      phi_val: final value of phi(lambda) (should be close to 0).
    """
    n = Q.shape[0]
    
    #  Compute the unconstrained minimizer: Q x + b = 0  =>  x = -Q^{-1}b.
    try:
        x_uncon = solve(Q, -b)
    except LinAlgError:
        raise ValueError("Matrix Q is singular or ill-conditioned.")

    # If the unconstrained solution is feasible, return it.
    if norm(x_uncon) <= Delta:
        return x_uncon, 0.0, 0, norm(x_uncon) - Delta

    # Define x(lambda) = -(Q + 2 lambda I)^{-1} b.
    def x_of_lambda(lam):
        return solve(Q + 2*lam*np.eye(n), -b)
    
    # Define phi(lambda) = ||x(lambda)|| - Delta.
    def phi(lam):
        return norm(x_of_lambda(lam)) - Delta

    # Step 1: Find an interval [lambda_low, lambda_high] where phi changes sign.
    lambda_low = 0.0
    lambda_high = 1.0
    while phi(lambda_high) > 0:
        lambda_high *= 2.0
        if lambda_high > 1e10:
            raise RuntimeError("Unable to bracket the root; problem may be ill-conditioned.")

    # Step 2: Bisection loop to find lambda such that ||x(lambda)|| ~= Delta.
    iter_count = 0
    lam_star = None
    x_star = None
    while iter_count < max_iter:
        lam_mid = 0.5 * (lambda_low + lambda_high)
        x_mid = x_of_lambda(lam_mid)
        # Update bracket based on the current norm.
        if norm(x_mid) > Delta:
            lambda_low = lam_mid
        else:
            lambda_high = lam_mid

        iter_count += 1
        
        # Check stopping criterion on lambda.
        if abs(lambda_high - lambda_low) < tol:
            lam_star = lam_mid
            x_star = x_mid
            break
    else:
        # If max_iter reached without convergence, return the last iterate.
        lam_star = lam_mid
        x_star = x_mid

    return x_star, lam_star, iter_count, phi(lam_star)

#Take KKT from last week
def check_KKT(Q, b, Delta, x, lam, tol=1e-6):
   
    # Check the KKT conditions for the trust-region subproblem.
    
    # KKT conditions:
    #   1) Stationarity: (Q + 2*lam I)x + b = 0.
    #   2) Complementarity: lam * (||x||^2 - Delta^2) = 0.
    #   3) Primal feasibility: ||x||^2 <= Delta^2.
    #   4) Dual feasibility: lam >= 0.
    
    # Returns a dictionary with residuals.
   
    stationarity_resid = norm((Q + 2*lam*np.eye(Q.shape[0])).dot(x) + b)
    comp_resid = abs(lam * (norm(x)**2 - Delta**2))
    primal_feas = (norm(x)**2 - Delta**2) <= tol  # should be <= 0 (or near 0)
    dual_feas = lam >= -tol  # allow slight numerical negative drift
    
    return {
        'stationarity': stationarity_resid,
        'complementarity': comp_resid,
        'primal_feasibility': primal_feas,
        'dual_feasibility': dual_feas
    }


# -------------------------------
# Construct various test instances
# -------------------------------
def run_tests():
    n_runs = 100
    dims = 2
    alpha = 1000.0
    Delta = 1.0  # Trust-region radius.
    
    unconstrained_runs = 0
    active_runs = 0
    iterations_list = []
    kkt_stationarity_list = []
    tests = []
    
    for i in range(n_runs):
        # Random starting point x_k from [2,25]^dims.
        x_k = np.random.uniform(2, 25, dims)
        # For f1, the quadratic model at x_k:
        Q = Hf1(x_k, alpha)
        b = df1(x_k, alpha)
        # The unconstrained minimizer is p = -x_k.
        p_uncon = -x_k
        norm_uncon = norm(p_uncon)
    
    
    # Test Case 1: Unconstrained solution feasible
    # Q positive definite, unconstrained minimizer inside the ball.
    Q1 = np.array([[2.0, 0.0],
                   [0.0, 2.0]])
    b1 = np.array([-1.0, -1.0])
    Delta1 = 1.5  # norm(unconstrained) ~0.707, so feasible.
    tests.append(('Feasible Unconstrained', Q1, b1, Delta1))
    
    # Test Case 2: Unconstrained solution infeasible (boundary active)
    # Q positive definite, unconstrained minimizer outside the ball.
    Q2 = np.array([[2.0, 0.0],
                   [0.0, 2.0]])
    b2 = np.array([-2.0, -2.0])
    Delta2 = 1.0  # unconstrained minimizer would be [1,1] with norm ~1.414 > 1.
    tests.append(('Active Constraint', Q2, b2, Delta2))
    
    # Test Case 3: Indefinite Q (simple example)
    Q3 = np.array([[1.0, 0.0],
                   [0.0, -1.0]])
    b3 = np.array([0.5, 0.5])
    Delta3 = 1.0
    tests.append(('Indefinite Q', Q3, b3, Delta3))
    
    # Test Case 4: Negative definite Q
    Q4 = -np.array([[1.0, 0.0],
                    [0.0, 1.0]])
    b4 = np.array([1.0, 1.0])
    Delta4 = 0.8  # unconstrained minimizer is [1,1] (norm ~1.414), so constraint active.
    tests.append(('Negative Definite Q', Q4, b4, Delta4))
    
    print("Running Test Cases:\n")
    for name, Q, b, Delta in tests:
        print(f"Test: {name}")
        try:
            x_star, lam_star, iters, phi_val = solve_sphere_constrained(Q, b, Delta, tol=1e-8)
        except Exception as e:
            print("  Error:", e)
            continue

        kkt = check_KKT(Q, b, Delta, x_star, lam_star)
        print("  x* =", x_star)
        print("  lambda* =", lam_star)
        print("  ||x*|| =", norm(x_star))
        print("  Iterations =", iters)
        print("  phi(lambda*) =", phi_val)
        print("  KKT residuals:")
        for key, val in kkt.items():
            print(f"    {key}: {val}")
        print("-"*50)
        
# Run all tests
#run_tests()
